In [ ]:
# 安裝 BERTopic 與中文分詞工具
!pip install bertopic jieba sentence-transformers pandas numpy

import pandas as pd
import numpy as np
import jieba
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
def read_timeline_xlsx(path):
    # 直接解析 xlsx 的 XML，避免缺少 openpyxl 時讀檔失敗。
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rows = []

    def cell_text(cell):
        inline = cell.find("a:is", ns)
        value = cell.find("a:v", ns)
        return "".join(inline.itertext()) if inline is not None else (value.text if value is not None else "")

    with zipfile.ZipFile(path) as zf:
        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        sheet_rows = root.find("a:sheetData", ns)
        header_map = {}

        for row_idx, row in enumerate(sheet_rows):
            row_values = {}
            for cell in row:
                ref = cell.attrib.get("r", "")
                col = "".join(ch for ch in ref if ch.isalpha())
                row_values[col] = cell_text(cell)

            if row_idx == 0:
                header_map = row_values
                continue

            rows.append({header_map.get(col, col): value for col, value in row_values.items()})

    return pd.DataFrame(rows)

# --- [功能 A: 資料深度清洗] ---
def preprocess_fb_data(series_list):
    """
    解決 ValueError: Unsupported input type: float 的關鍵步驟。
    確保所有輸入皆為字串且非空。
    """
    df = pd.concat(series_list).dropna().astype(str).reset_index(drop=True)
    # 移除空白字串與過短的內容 (雜訊)
    df = df[df.str.strip().str.len() > 5].reset_index(drop=True)
    return df

# --- [功能 B: 中文分詞配置] ---
def tokenize_chinese(text):
    """
    針對 BERTopic 的 c-TF-IDF 進行中文分詞適配
    """
    words = jieba.lcut(re.sub(r'[^\u4e00-\u9fa5]', '', text))
    return " ".join([w for w in words if len(w) > 1])

# --- [功能 C: 初始化模型元件] ---
def get_bertopic_components():
    # 1. Embedding: 使用多國語言模型 (支援繁體中文)
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # 2. UMAP: 降維並保留局部拓撲結構
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
    
    # 3. HDBSCAN: 自動發現聚類 (不預設 K 值)
    hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', 
                            cluster_selection_method='eom', prediction_data=True)
    
    # 4. Vectorizer: 處理中文分詞
    vectorizer_model = CountVectorizer(stop_words=None) # 停用詞可後續加入
    
    return embedding_model, umap_model, hdbscan_model, vectorizer_model

In [ ]:
# 1. 載入資料 (請替換為你的 FB 資料)
# df_all = preprocess_fb_data([fb_groups_series, fb_posts_series])
# 模擬測試
posts = read_timeline_xlsx("time_line.xlsx")
text_columns = ["content", "share_content", "attachment_title", "attachment_description"]
posts[text_columns] = posts[text_columns].fillna("").astype(str)
posts["text"] = posts[text_columns].agg(" ".join, axis=1)
df_docs = posts["text"]
cleaned_docs = preprocess_fb_data([df_docs])

# 2. 中文預分詞 (BERTopic 處理中文的必要步驟)
processed_docs = cleaned_docs.apply(tokenize_chinese).tolist()

# 3. 建立並訓練模型
embed_m, umap_m, hdb_m, vec_m = get_bertopic_components()

topic_model = BERTopic(
    embedding_model=embed_m,
    umap_model=umap_m,
    hdbscan_model=hdb_m,
    vectorizer_model=vec_m,
    verbose=True
)

topics, probs = topic_model.fit_transform(processed_docs)

# 4. 提取結果：這就是你想要的「不好的概念」與「錨點詞」
topic_info = topic_model.get_topic_info()
print("\n>>> 自動發現的語意主題列表：")
print(topic_info.head(10))

# 儲存每個主題的前 20 個關鍵詞作為子空間錨點
topic_info.to_csv("malicious_concepts_taxonomy.csv", index=False)

In [ ]:
# 視覺化主題分布
topic_model.visualize_topics()

# 針對特定句子進行解構分析 (投影到子空間)
test_sentence = "保證獲利，現在入金立享回饋"
test_processed = tokenize_chinese(test_sentence)
target_topic, prob = topic_model.transform([test_processed])

print(f"\n句子: '{test_sentence}'")
print(f"被歸類的主題 ID: {target_topic[0]}")
print(f"在該子空間的分量強度 (Confidence): {prob[0]:.4f}")